# QUOPS Circuits in pytket



Here we present tools to create pytket circuits required to perform the QUOPS benchmark. Later in this notebook we will also introduce methods for processing the results of running these circuit. We will assume that the reader is familiar with the theoretical foundations of the QUOPS benchmark, and aim only to introduce its implementation in pytket. For a complete description of the QUOPS benchmark we refer the reader to the paper and/or the guppy notebook which is more extensive. Note also that the methods used here are for demonstration purposes and do not necessarily align with the exact method used in the results presented in the paper. We direct the reader to the paper for full details on the number of shots, the number of circuits, etc used there.

The cell below constructs a QUOPS circuit, consisting of random single qubit gates and entangling gates between random pairs of qubits. Note that, while we do not do so in this notebook, you may wish to optimise the QUOPS circuit generated before proceeding. In the results presented in the paper we use default pytket compiler passes to do this.

In [27]:
from pytket import Circuit, OpType
import numpy as np
from pytket.circuit.display import render_circuit_jupyter
from numpy.random import Generator

def random_qv2_circuit(
    n_qubits: int,
    depth: int | float,
    rng: Generator = np.random.default_rng(),
) -> Circuit:
    """
    Create pytket circuit implementing QUOPS circuit. Consists of
    alternating layers of 
        (i) random single-qubit unitaries of the form exp[- i (theta/2) P] 
            for P in {X, Y, Z}, where both theta and P are chosen uniformly
            at random, and 
        (ii) CXs between pairs of qubits given a random qubit pairing.

    Generator `rng` is used for both random qubit permutations and drawing
    random single-qubit rotations.

    Parameters
    ----------
    n_qubits: int
        Number of qubits
    depth: int | float
        Number of layers. If not an integer, will add a fractional layer at
        the end of the circuit.
    rng: np.random.Generator
        Random number generator for all circuit randomization

    Returns
    -------
    Circuit
        A random QUOPS circuit
    """

    circuit = Circuit(n_qubits=n_qubits)
    gate_names = [OpType.Rx, OpType.Ry, OpType.Rz]

    depth_bulk = int(depth)

    for _ in range(depth_bulk):
        # Draw random unitaries
        for qubit in range(circuit.n_qubits):
            random_angle = rng.uniform(0, 4)
            random_gate = rng.choice(gate_names)
            # Add random Pauli rotation gate to circuit
            circuit.add_gate(random_gate, [random_angle], [qubit])

        # Construct random permutation of qubits
        qubits = rng.permutation(circuit.n_qubits)
        qubit_pairs = [pair for pair in zip(qubits[::2], qubits[1::2])] # leave out unpaired qubit

        # Add CX's between randomly paired qubits
        for control, target in qubit_pairs:
            circuit.CX(control, target)

    # Add final partial layer
    partial = depth - depth_bulk
    n_partial_layer = round(partial * n_qubits)  # Qubits acted on in partial layer

    if n_partial_layer > 0:
        # Choose `n_partial_layer` qubits from all qubits without replacement
        partial_qubit_indices = rng.permutation(np.arange(n_qubits))[:n_partial_layer]

        for qubit in partial_qubit_indices:
            random_angle = rng.uniform(0, 4)
            random_gate = rng.choice(gate_names)
            # Add random Pauli rotation gate to circuit
            circuit.add_gate(random_gate, [random_angle], [qubit])

        qubit_pairs = [pair for pair in zip(partial_qubit_indices[::2], partial_qubit_indices[1::2])] # leave out unpaired qubit

        # Add CX's between randomly paired qubits
        for control, target in qubit_pairs:
            circuit.CX(control, target)

    return circuit

circuit = random_qv2_circuit(n_qubits=3, depth=2)
render_circuit_jupyter(circuit)


To measure the polarisation of the circuit we will use mirror benchmarking. As such in the following we mirror the circuit generated above. We will also need to rebase the circuit into the Quantinuum native gate-set. A barrier is added to prevent optimisation between the circuit and its mirror. The mirror will be appended to the original circuit a few cells down, after some more transformation have been applied.

In [28]:
from pytket.passes import AutoRebase
from copy import deepcopy

mirror = deepcopy(circuit).dagger()
circuit.add_barrier(circuit.qubits)

AutoRebase({OpType.ZZMax, OpType.Rz, OpType.PhasedX}).apply(circuit)
AutoRebase({OpType.ZZMax, OpType.Rz, OpType.PhasedX}).apply(mirror)

render_circuit_jupyter(circuit)

One approach to measuring polarisation uses three circuit types.
- Mirrored circuits where randomised compiling is applied to only one half of the mirrored circuit.
- Mirrored circuits where randomised compiling is applied to both halves.
- Circuits measuring SPAM error rates.

We describe how to construct the half randomised circuit, although the fully randomised circuits can straightforwardly be constructed using the same tools. 

In the following we use a compilation pass which applies randomised compiling to each of the ZZmax gates.

In [29]:
from quops_with_pytket_randomisation import ZZMaxRandomCompilation
from quops_with_pytket_randomisation import RandomnessType

mirror_copy = deepcopy(mirror)

ZZMaxRandomCompilation(randomness_type=RandomnessType.PER_SHOT).apply(mirror_copy)

render_circuit_jupyter(mirror_copy)

This initialises classical registers with random values which are used to control Pauli gates acting before each ZZMax gate. Compensating Pauli gates are then applied after the ZZMax to preserve the operation being performed. 

Note that we have used `randomness_type=RandomnessType.PER_SHOT` here. This ensures that the randomness used to control the Paulis differs between each shot. This functionality is supported on Quantinuum machines. However other machines may not support this, in which case `RandomnessType.NO_CLASSICAL` should be used. In this case the Pauli gates will be applied randomly, but will be identical for each shot. As such, several such circuits will need to be created to mimic randomised compiling. We will use `RandomnessType.NO_CLASSICAL` in the remainder of this notebook as the resulting circuits are easier to display. However `RandomnessType.PER_SHOT` is supported for all of the randomisation passes presented here.

We additionally append the randomised compiled circuit to the original circuit to create the mirrored circuit.

In [30]:
from quops_with_pytket_randomisation import ZZMaxRandomCompilation
from quops_with_pytket_randomisation import RandomnessType

ZZMaxRandomCompilation(randomness_type=RandomnessType.NO_CLASSICAL).apply(mirror)

render_circuit_jupyter(mirror)

circuit.append(mirror)

In the cell below we perform frame randomisation by prepending random single-qubits Cliffords to the circuit, and appending the inverse gates. We will need to rebase the circuit again as the clifford gates added are not necessarily in the correct gate-set.

In [31]:
from quops_with_pytket_randomisation import FrameRandomisation

FrameRandomisation(randomness_type=RandomnessType.NO_CLASSICAL).apply(circuit)
render_circuit_jupyter(circuit)

AutoRebase({OpType.ZZPhase, OpType.ZZMax, OpType.Rz, OpType.PhasedX}).apply(circuit)

True

Now we can measure the circuit.

In [32]:
measurement_register = circuit.add_c_register(
    name="qubit_measurements",
    size=circuit.n_qubits,
)
measurement_cbits = measurement_register.to_list()

for qubit, cbit in zip(circuit.qubits, measurement_cbits, strict=True):
    circuit.Measure(qubit, cbit)

render_circuit_jupyter(circuit)

You may optionally wish to perform leakage error detection. This will add an ancillary qubit for each qubit. The ancillary qubit measures if a leakage event has occurred. `n_device_qubits` can be used to specify how many qubits are available on the device you are using. Here we use 6 as this is the number needed to ensure each leakage detection gadget has its own qubit to use. However if fewer qubits are available then some will be reset and reused as leakage detection ancillary after they have been measured.

In [33]:
from pytket.extensions.quantinuum.backends.leakage_gadget import get_detection_circuit

circuit = get_detection_circuit(circuit=circuit, n_device_qubits=6)
render_circuit_jupyter(circuit)

The final modification required is to randomly flip the measurement outcomes. This can be done using `RandomXPass` which will apply random X gates before the measurements, and will perform a compensating classical NOT on the classical registers containing the measurement results.

In [34]:
from quops_with_pytket_randomisation import RandomXPass

RandomXPass(randomness_type=RandomnessType.NO_CLASSICAL).apply(circuit)
render_circuit_jupyter(circuit)

As expected, this circuit generates all 0 measurement outcomes in the case that there is no noise.

In [35]:
from pytket.extensions.quantinuum import QuantinuumAPIOffline, QuantinuumBackend

backend = QuantinuumBackend(
    device_name="H1-1LE",
    api_handler=QuantinuumAPIOffline(),
)
compiled_circuit = backend.get_compiled_circuit(circuit, optimisation_level=0)
measurement_cbits = circuit.get_c_register("qubit_measurements").to_list()
result = backend.run_circuit(compiled_circuit, n_shots=100)
result.get_counts(cbits=measurement_cbits)

Counter({(0, 0, 0): 100})

## Minimal local QUOPS experiment

The cells below run a small QUOPS experiment on the Quantinuum local emulator, and estimate the polarisation with a bootstrap lower confidence bound. 

First, `build_measured_quops_circuit` wraps up the discussion we conducted above into a convenience method. 

In [36]:
from collections import Counter
from copy import deepcopy
from pytket.extensions.quantinuum import QuantinuumAPIOffline, QuantinuumBackend

quops_width = 3
quops_depth = 2
quops_repetitions = 6
quops_shots_per_circuit = 100

def build_measured_quops_circuit(width: int, depth: int, randomize_both: bool, seed: int):
    rng = np.random.default_rng(seed)
    forward = random_qv2_circuit(n_qubits=width, depth=depth, rng=rng)
    mirror = deepcopy(forward).dagger()
    forward.add_barrier(forward.qubits)

    AutoRebase({OpType.ZZMax, OpType.Rz, OpType.PhasedX}).apply(forward)
    AutoRebase({OpType.ZZMax, OpType.Rz, OpType.PhasedX}).apply(mirror)

    if randomize_both:
        forward.append(mirror)
        ZZMaxRandomCompilation(
            rng=rng,
            randomness_type=RandomnessType.NO_CLASSICAL,
        ).apply(forward)
    else:
        ZZMaxRandomCompilation(
            rng=rng,
            randomness_type=RandomnessType.NO_CLASSICAL,
        ).apply(mirror)
        forward.append(mirror)

    FrameRandomisation(
        rng=rng,
        randomness_type=RandomnessType.NO_CLASSICAL,
    ).apply(forward)
    AutoRebase({OpType.ZZPhase, OpType.ZZMax, OpType.Rz, OpType.PhasedX}).apply(forward)

    measurement_register = forward.add_c_register("qubit_measurements", forward.n_qubits)
    measurement_cbits = measurement_register.to_list()
    for qubit, cbit in zip(forward.qubits, measurement_cbits, strict=True):
        forward.Measure(qubit, cbit)

    RandomXPass(
        rng=rng,
        randomness_type=RandomnessType.NO_CLASSICAL,
    ).apply(forward)
    return forward, measurement_cbits

def project_counts(result, cbits):
    return Counter(
        {
            "".join(str(bit) for bit in outcome): count
            for outcome, count in result.get_counts(cbits=cbits).items()
        }
    )

Of note is that the SPAM circuit can be simply created by creating a depth 0 circuit.

In [37]:
spam_circuit, spam_cbits = build_measured_quops_circuit(
    width=quops_width,
    depth=0,
    randomize_both=False,
    seed=29,
)
render_circuit_jupyter(spam_circuit)

In the following we run a selection of these circuits. Note that we are using an ideal local simulator here for demonstration purposes, but as a result the results will be that the polarisation is 1.

In [38]:
def run_local_quops_experiment(width: int, depth: int, repetitions: int, n_shots: int, seed: int = 1000):
    backend = QuantinuumBackend("H1-1LE", api_handler=QuantinuumAPIOffline())
    role_counts = {"both": [], "half": [], "spam": []}

    for rep in range(repetitions):
        both_circuit, both_cbits = build_measured_quops_circuit(
            width=width,
            depth=depth,
            randomize_both=True,
            seed=seed + rep,
        )
        half_circuit, half_cbits = build_measured_quops_circuit(
            width=width,
            depth=depth,
            randomize_both=False,
            seed=seed + repetitions + rep,
        )

        both_result = backend.run_circuit(
            backend.get_compiled_circuit(both_circuit, optimisation_level=0),
            n_shots=n_shots,
        )
        half_result = backend.run_circuit(
            backend.get_compiled_circuit(half_circuit, optimisation_level=0),
            n_shots=n_shots,
        )

        role_counts["both"].append(project_counts(both_result, both_cbits))
        role_counts["half"].append(project_counts(half_result, half_cbits))

    spam_circuit, spam_cbits = build_measured_quops_circuit(
        width=width,
        depth=0,
        randomize_both=False,
        seed=seed + 2 * repetitions,
    )
    spam_result = backend.run_circuit(
        backend.get_compiled_circuit(spam_circuit, optimisation_level=0),
        n_shots=n_shots * repetitions,
    )
    role_counts["spam"].append(project_counts(spam_result, spam_cbits))
    return role_counts

quops_results = run_local_quops_experiment(
    width=quops_width,
    depth=quops_depth,
    repetitions=quops_repetitions,
    n_shots=quops_shots_per_circuit,
)
{
    role: {
        "circuits": len(counters),
        "shots": sum(counter.total() for counter in counters),
    }
    for role, counters in quops_results.items()
}

{'both': {'circuits': 6, 'shots': 600},
 'half': {'circuits': 6, 'shots': 600},
 'spam': {'circuits': 1, 'shots': 600}}

In the final cell we convert the measured bitstring counts for the `both`, `half`, and `spam` circuit families into Hamming-weight samples, and use those samples to estimate the observed polarisation of each family. The QUOPS estimate is then formed as $P_{\mathrm{half}} / \sqrt{P_{\mathrm{both}} P_{\mathrm{spam}}}$, which removes the SPAM contribution from the half-randomised data.

This cell also computes a simple bootstrap lower confidence bound. Each bootstrap sample resamples the observed shots with replacement within the three circuit families, recomputes the QUOPS polarisation, and stores the result. The reported `lower_confidence` is the 5th percentile of those bootstrap values.

In [39]:
def counts_to_hamming_weights(counts):
    return np.array(
        [
            bitstring.count("1")
            for bitstring, count in counts.items()
            for _ in range(count)
        ],
        dtype=int,
    )

def observed_polarisation(hamming_weights, n_qubits: int):
    hamming_weights = np.asarray(hamming_weights, dtype=float)
    summation = np.mean((-0.5) ** hamming_weights)
    scale = (4 ** n_qubits) / (4 ** n_qubits - 1)
    offset = 1 / (4 ** n_qubits - 1)
    return float(scale * summation - offset)

def total_polarisation(both_weights, half_weights, spam_weights, n_qubits: int):
    both_pol = observed_polarisation(both_weights, n_qubits)
    half_pol = observed_polarisation(half_weights, n_qubits)
    spam_pol = observed_polarisation(spam_weights, n_qubits)
    return half_pol / np.sqrt(both_pol * spam_pol)

def polarisation_with_lower_confidence(role_counts, n_qubits: int, n_bootstrap: int = 2000, seed: int = 1234):
    both_weights = np.concatenate([counts_to_hamming_weights(counts) for counts in role_counts["both"]])
    half_weights = np.concatenate([counts_to_hamming_weights(counts) for counts in role_counts["half"]])
    spam_weights = np.concatenate([counts_to_hamming_weights(counts) for counts in role_counts["spam"]])

    estimate = total_polarisation(both_weights, half_weights, spam_weights, n_qubits)
    rng = np.random.default_rng(seed)
    bootstrap_samples = np.empty(n_bootstrap, dtype=float)

    for index in range(n_bootstrap):
        bootstrap_samples[index] = total_polarisation(
            rng.choice(both_weights, size=both_weights.size, replace=True),
            rng.choice(half_weights, size=half_weights.size, replace=True),
            rng.choice(spam_weights, size=spam_weights.size, replace=True),
            n_qubits,
        )

    return {
        "polarisation": float(estimate),
        "lower_confidence": float(np.quantile(bootstrap_samples, 0.05)),
    }

quops_stats = polarisation_with_lower_confidence(quops_results, quops_width)
quops_stats

{'polarisation': 1.0, 'lower_confidence': 1.0}

Note that in the experiments presented in the QUOPS paper the experiment on H2-1 use a MCFE procedure in which both halves of the mirror circuit employ Pauli frame randomisation. This approach reduces the sample complexity of the experiments. However as the circuits used are a subset of the circuits used in the experiment described above, we have presented the higher sample complexity approach here for completeness.